# Building our model using XGboost

## Importing Packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
%pip install xgboost
from xgboost import XGBClassifier

## Importing data

In [ ]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/processed/cleaned_train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/processed/cleaned_train.csv'
    },
    'test': {
        'local': '../data/processed/cleaned_test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/processed/cleaned_test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }
    
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

## Setting up our columns

In [ ]:
target_col = 'cost_category'
id_col = 'Tour_ID'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']

# Map string labels to numeric integers (0 to 5) for multi-class training
class_to_idx = {cls_name: i for i, cls_name in enumerate(target_classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(target_classes)}
train['target'] = train[target_col].map(class_to_idx)

## Categorical Feature Specification

In [ ]:
cat_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity', 
    'info_source', 'tour_arrangement', 'package_transport_int', 
    'package_accomodation', 'package_food', 'package_transport_tz', 
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz'
]

train_df = train.copy()
test_df = test.copy()

# XGBoost requires categorical columns to be pandas 'category' dtype
for col in cat_features:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

features = [col for col in train_df.columns if col not in [id_col, target_col, 'target']]

X = train_df[features]
y = train_df['target']
X_test = test_df[features]

## Stratified 5-Fold Cross-Validation with XGBoost

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(train_df), len(target_classes)))
test_preds = np.zeros((len(test_df), len(target_classes)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    model = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.04,
        max_depth=6,
        objective='multi:softprob',
        num_class=len(target_classes),
        eval_metric='mlogloss',
        enable_categorical=True,  # Enables native handling of categorical features
        tree_method='hist',       # Required for native categorical support
        random_state=42,
        early_stopping_rounds=100
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=200
    )
    
    # Store Out-of-Fold predictions and test predictions
    oof_preds[val_idx] = model.predict_proba(X_va)
    test_preds += model.predict_proba(X_test) / skf.n_splits

# Overall Out-Of-Fold Multi-Class Log Loss Metric
cv_log_loss = log_loss(y, oof_preds)
print(f"\n==========================================")
print(f"Overall OOF Log Loss: {cv_log_loss:.5f}")
print(f"==========================================")

## XGBoost Submission

In [ ]:
submission = pd.DataFrame(test_preds, columns=[idx_to_class[i] for i in range(len(target_classes))])
submission.insert(0, id_col, test_df[id_col])

# Reorder columns to strictly match SampleSubmission.csv
target_order = ['Tour_ID', 'High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
submission = submission[target_order]

submission.to_csv('xgboost_submission.csv', index=False)
print("Saved predictions to xgboost_submission.csv")